In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

In [2]:
# load the datasets
train = pd.read_csv("fashion-mnist_train.csv")
test = pd.read_csv("fashion-mnist_test.csv")

print(train.shape)
print(test.shape)

(60000, 785)
(10000, 785)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
X_train = train.drop('label' , axis=1)
y_train = train['label']
X_test = test.drop('label' , axis=1)
y_test = test['label']

In [5]:
# convert to tensors
X_train = torch.from_numpy(np.array(X_train)).to(dtype=torch.float32)
X_test = torch.from_numpy(np.array(X_test)).to(dtype=torch.float32)
y_train = torch.from_numpy(np.array(y_train)).to(dtype=torch.long)
y_test = torch.from_numpy(np.array(y_test)).to(dtype=torch.long)

In [6]:
# standardize the data from 0 to 1

X_train = X_train/255.
X_test = X_test/255.

In [21]:
# create dataset and dataloader
from torch.utils.data import Dataset , DataLoader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return self.features[idx] , self.labels[idx]

In [22]:
train_dataset = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test , y_test)

In [23]:
train_loader = DataLoader(train_dataset , batch_size=256 , shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size=256 , shuffle=True,pin_memory=True)

**The **

In [28]:
## Model Building
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self , num_features):

        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features , 128),
            nn.BatchNorm1d(128), # added batch normalization 
            nn.ReLU(),
            nn.Dropout(p=0.4), # added dropouts with prob = 0.4

            nn.Linear(128 , 64),
            nn.BatchNorm1d(64), 
            nn.ReLU(),
            nn.Dropout(p=0.4),
            
            nn.Linear(64,10)
        )

    # forward pass

    def forward(self,features):
        out = self.network(features)
        return out

In [29]:
# Parameters
learning_rate = 0.1
epochs=100

# create the model Object
model = MyModel(X_train.shape[1])
model = model.to(device) # this saves my model in GPU

# create loss

criterion = nn.CrossEntropyLoss()

# optimizer

optimizer = optim.SGD(model.parameters() ,weight_decay= 1e-4, lr = learning_rate)

In [30]:
for batch_features, batch_labels in train_loader:
    print(batch_features.shape)
    print(batch_labels.shape)
    break

torch.Size([256, 784])
torch.Size([256])


In [ ]:
# Let us now build the back propagation code
import time
start_time = time.time()
for epoch in range(epochs):
    total_loss = 0
    for batch_features,batch_labels in train_loader:
        # save them in GPU
        batch_features,batch_labels = batch_features.to(device), batch_labels.to(device)

        # forward pass

        outputs = model(batch_features)

        # calc loss

        loss = criterion(outputs , batch_labels) # this automatically converts logits to probs using softmax

        # remove the gradients
        optimizer.zero_grad()

        # back propagate
        loss.backward()

        # update the weights
        optimizer.step()

        # find the sum of loss in each epoch
        total_loss += loss.item()
    print(f'Loss at epoch {epoch+1} ->{total_loss/len(train_loader)}')
print(time.time()-start_time)

Loss at epoch 1 ->0.7771716713905334
Loss at epoch 2 ->0.5403976440429688
Loss at epoch 3 ->0.4952113926410675
Loss at epoch 4 ->0.46705949306488037
Loss at epoch 5 ->0.4445415735244751
Loss at epoch 6 ->0.4316795766353607
Loss at epoch 7 ->0.41602298617362976
Loss at epoch 8 ->0.40857210755348206
Loss at epoch 9 ->0.39989522099494934
Loss at epoch 10 ->0.3906971514225006
Loss at epoch 11 ->0.3824431002140045
Loss at epoch 12 ->0.3781166076660156
Loss at epoch 13 ->0.36986634135246277
Loss at epoch 14 ->0.3644595146179199
Loss at epoch 15 ->0.3626299202442169
Loss at epoch 16 ->0.35805732011795044
Loss at epoch 17 ->0.35491812229156494
Loss at epoch 18 ->0.34728631377220154
Loss at epoch 19 ->0.34462636709213257
Loss at epoch 20 ->0.34114518761634827
Loss at epoch 21 ->0.3381710946559906
Loss at epoch 22 ->0.33453282713890076
Loss at epoch 23 ->0.32977092266082764
Loss at epoch 24 ->0.3267032504081726
Loss at epoch 25 ->0.32032716274261475
Loss at epoch 26 ->0.3212679624557495
Loss at 

In [37]:
# Evaluation 
# set to eval mode
model.eval()
total = 0
correct = 0
for batch_features , batch_labels in test_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')



Accuracy Score -> 0.8925


In [38]:
# Evaluation of training data
# set to eval mode
model.eval()
total = 0
correct = 0
for batch_features , batch_labels in train_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')



Accuracy Score -> 0.9420333333333333


- We can see that our model is overfitting, to reduce overfitting we can perform a few things:
    - 1. Introduce dropouts after each layer.
    - 2. Use BatchNormalization in each layer to normalize the output data of every hidden layer
    - 3. Third is to use regularization (L2) using weight_decay